### Evaluation + Exploration of the LangChain Chatbot Output
This notebook compares the output on various queries to make sure the langchain model is functioning correctly and incorporating API context, not just returning information directly from openai.</br></br>

In [1]:
#IMPORTS
'''Specific versions of LangChain needed to run Ragas - different from needed for LangChain for Chatbot'''
#! pip install langchain-openai==0.0.2
#! pip install langchain-core==0.1.45
import time
import sys
sys.path.insert(0,'../')
from environment import env
config = env.env()

OPENAI_API_KEY = config['gpt_api_key']
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

from langchain_openai import ChatOpenAI
from langchain.schema.messages import HumanMessage, SystemMessage
#from langchain.memory.buffer import ConversationBufferMemory
#from langchain.prompts import (
#    ChatPromptTemplate,
#)"""

from model_functions.gnb_model import *
from model_functions.get_context import *
from model_functions.get_context import parsed_context
from model_functions.lang_chatbot import nps_chain

import nest_asyncio
import openai
openai.api_key = OPENAI_API_KEY
nest_asyncio.apply()

#from langchain.chains import RetrievalQA
from ragas.metrics import answer_similarity
from ragas import evaluate
from datasets import Dataset


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/lauralyns/Documents/MADS/SIADS 699 -
[nltk_data]     Capstone/Capstone
[nltk_data]     VS/MADS_Capstone/.venv/lib/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lauralyns/Documents/MADS/SIADS 699 -
[nltk_data]     Capstone/Capstone
[nltk_data]     VS/MADS_Capstone/.venv/lib/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/lauralyns/Documents/MADS/SIADS 699 -
[nltk_data]     Capstone/Capstone
[nltk_data]     VS/MADS_Capstone/.venv/lib/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


'\nfrom langchain.chains import RetrievalQA\nfrom ragas.metrics import answer_similarity\nfrom ragas import evaluate\n#from ragas.langchain.evalchain import RagasEvaluatorChain\nfrom ragas.metrics import (\n    faithfulness,\n    answer_relevancy,\n    context_precision,\n    context_recall,\n)\nfrom datasets import Dataset\n'

In [2]:
#VARIABLES
model = gnb_model
chat_model = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0,api_key  = OPENAI_API_KEY)

In [217]:
#costs
gpt3 = .03/20
embed = .01/28
gpt3 *4000 + embed*2000

pre_calls = 25.38
gpt_100 = 25.39 - 25.38
lang_100 = 25.48 - 25.39

gpt_lang_100 = 25.59 - 25.39

gpt_100 = 0.0
lang_100 = 25.62 - 25.59

gpt_200 = 25.72 - 25.59
lang_200 = 25.90 - 25.72


gpt_lang_250 = 26.21 - 25.90
gpt_lang_to_1100 = 26.71 - 26.21

miscalculate = 27.24 - 26.74

gpt_to_2000ish = - 27.24

6.714285714285714

In [2]:
#Time for calls
test_100 = 50 * 60
break_len = 1000 #seconds
timing_100 = test_100  - break_len * 3
timing_100
#5 min?

0

#### Chat Models
Defining Chat models to be easily invoked for this testing context

In [18]:
def gpt_chat(query, intent="", endpoint="", parkcode=""):
    """Takes Question as input and returns the result of running it through the generic chat_model used for reference
    intent, endpoint and parkcode included so that it can be invoked in same way as other models"""
    messages = [
        SystemMessage(content="""You're an assistant knowledgeable about national parks. Only answer national park related questions."""),
        HumanMessage(content=query),]
    return chat_model.invoke(messages).content

In [19]:
gpt_chat("What is yosemite like")

"Yosemite National Park is located in California's Sierra Nevada mountains and is known for its stunning granite cliffs, waterfalls, giant sequoia trees, and diverse wildlife. The park offers a variety of outdoor activities such as hiking, rock climbing, camping, and wildlife viewing. Yosemite Valley is a popular destination within the park, known for iconic landmarks like El Capitan and Half Dome."

In [12]:
def lang_chat(query, intent, endpoint, parkcode):
    """Takes Question as input and returns the result of running it through LangChain promp templat chat model (nps_chain.py) 
    Instead of using get_context(question, model) for context as usual, the correct API landing page is handed through as context with 
    call_api() function which uses intent, endpoint and parkcode from test data to ensure that the correct context is referenced. 
    Returns model results"""
    return nps_chain.invoke({"context": call_api(endpoint, parkcode, intent), "question": query})

In [13]:
lang_chat("What is yosemite like", "description", "parks", "yose")

'Yosemite National Park is not just a great valley, but a shrine to human foresight, the strength of granite, the power of glaciers, the persistence of life, and the tranquility of the High Sierra. It is best known for its waterfalls, but within its nearly 1,200 square miles, you can find deep valleys, grand meadows, ancient giant sequoias, a vast wilderness area, and much more.'

In [15]:
def ground_truth(query, intent, endpoint, parkcode):
    """Takes in the intent, endpoint and parkcode from test data and runs through parsed_context() to find context from NPS webpage
    Returns the ground_truth context from the correct API call, to be compared against by the other models"""
    return parsed_context(endpoint, parkcode, intent)

In [16]:
ground_truth('', 'description', 'parks', 'yose')

'Not just a great valley, but a shrine to human foresight, the strength of granite, the power of glaciers, the persistence of life, and the tranquility of the High Sierra. First protected in 1864, Yosemite National Park is best known for its waterfalls, but within its nearly 1,200 square miles, you can find deep valleys, grand meadows, ancient giant sequoias, a vast wilderness area, and much more.'

In [6]:
models = {
    'gpt_chat' : gpt_chat,
    'lang_chat': lang_chat,
    'ground_truth': ground_truth,
}

##### Test Data
Constructing Test Data set, and making sure it is representative

In [7]:
test_data = pd.read_csv('../02_nps_api_data/testing_queries.csv')
test_data.columns = ['query', 'intent', 'endpoint', 'parkcode']
test_data = test_data.sample(3000, random_state=42)
test_data.head()

,query,intent,endpoint,parkcode
27641,What are the fees for visiting Virgin Islands ...,feespass,feespasses,vicr
15777,What is available at Denali National Park & Pr...,amenities,amenities,dena
24454,What events are scheduled at Lower Delaware?,events,events,lode
6328,What is the address of Saint Paul's Church,address,parks,sapa
28771,What are the fees for visiting Grant-Kohrs Ranch,feespass,feespasses,grko


In [ ]:
#compare % of each for test data versus sample of data?

In [8]:
test_data['intent'].value_counts()

intent
amenities      582
events         499
alerts         476
feespass       438
description    353
fullname       296
address        264
state           92
Name: count, dtype: int64

In [9]:
test_data['endpoint'].value_counts()

endpoint
parks         1005
amenities      582
events         499
alerts         476
feespasses     438
Name: count, dtype: int64

In [10]:
test_data['parkcode'].value_counts()

parkcode
lowe    15
fopu    14
wwii    13
cahi    12
foun    12
        ..
arpo     2
kaww     2
flni     2
mana     1
misp     1
Name: count, Length: 471, dtype: int64

#### Comparing Context to Results

In [11]:
def add_responses(query_df, chat_models, n=0):
    """Adds the responses for given chat models to the dataframe of questions, intents, parkcodes and queries, and saves as pickle"""
    while n < len(query_df):
        segment_time = []
        segment_df = query_df[n:n+400]

        for model in chat_models:
            start = time.time()
            temp_df = segment_df.apply(lambda x: chat_models[model](x['query'],x['intent'],x['endpoint'],x['parkcode']), axis=1)
            stop = time.time()

            temp_df = pd.DataFrame(temp_df, columns = [model])
            segment_df = pd.concat([segment_df, temp_df], axis=1)
            segment_df = segment_df.fillna('none')
        
            segment_time.append((stop-start)/len(segment_df))
            #Sleeps for 1 hr so that NPS can be called again and does not go over NPS API limit of 1000 requests / hour
            #time.sleep(3600)
     
        if n ==0:
            test_df = segment_df
            times = pd.DataFrame(columns=chat_models.keys())
        else:
            test_df = pd.read_pickle('chat_test.pkl')
            test_df = pd.concat([test_df, segment_df], axis=0)
            times = pd.read_pickle('chat_times.pkl')
            
        times.loc[n] = segment_time
        test_df.to_pickle('chat_test.pkl')
        times.to_pickle('chat_times.pkl')
        n=n+400
        print(n, 'completed')

In [29]:
#Uncomment to run if chat_test.pkl does not exist. Otherwise takes x hours and costs ~$x money to run per k of data
#add_responses(test_data[0:2000], models, 1101)

In [ ]:
#Completed:
#0-100
#100-200
#200-400
#400-650
#650-1100

In [12]:
responses = pd.read_pickle('chat_test.pkl')
responses

,query,intent,endpoint,parkcode,gpt_chat,lang_chat,ground_truth
27641,What are the fees for visiting Virgin Islands ...,feespass,feespasses,vicr,There are no entrance fees for visiting Virgin...,Virgin Islands Coral Reef National Monument is...,Interagency Pass is not accepted.
15777,What is available at Denali National Park & Pr...,amenities,amenities,dena,"Denali National Park & Preserve, located in Al...",Denali National Park & Preserve offers a varie...,"Accessible Rooms, Assistive Listening Systems,..."
24454,What events are scheduled at Lower Delaware?,events,events,lode,"I'm sorry, but there is no national park calle...",Lower Delaware National Wild and Scenic River ...,No events information found
6328,What is the address of Saint Paul's Church,address,parks,sapa,"I'm sorry, I can only provide information abou...",The address of Saint Paul's Church National Hi...,"897 South Columbus Avenue, Mount Vernon, NY, 1..."
28771,What are the fees for visiting Grant-Kohrs Ranch,feespass,feespasses,grko,The entrance fee for Grant-Kohrs Ranch Nationa...,"Grant-Kohrs Ranch is a fee-free park, so there...",Interagency Pass is not accepted.
...,...,...,...,...,...,...,...
30025,How much is a ticket for Washington Monument?,feespass,feespasses,wamo,The Washington Monument is not a national park...,A ticket for the Washington Monument costs $1.00.,Interagency Pass is not accepted. There is no ...
20117,What services are there at Richmond Park?,amenities,amenities,rich,Richmond Park is a royal park located in Londo...,Richmond Park is a national park located in Lo...,No amenities information found
1660,Give me information on Theodore Roosevelt Inau...,description,parks,thri,Theodore Roosevelt Inaugural National Historic...,Theodore Roosevelt Inaugural National Historic...,"As president, Theodore Roosevelt created prote..."
499,Give me information on Edgar Allan Poe Nationa...,description,parks,edal,"I'm sorry, but there is no national park calle...",The Edgar Allan Poe National Historic Site is ...,"Described as horrifying, mystifying, and brill..."


In [23]:
len(responses)

1100

In [24]:
responses.duplicated(subset=['query']).value_counts()

False    1097
True        3
Name: count, dtype: int64

In [15]:
responses[responses.duplicated(subset=['query'],keep=False)]

,query,intent,endpoint,parkcode,gpt_chat,lang_chat,ground_truth
16565,What services are there at Kenai Fjords Nation...,amenities,amenities,kefj,"At Kenai Fjords National Park, visitors can en...",Kenai Fjords National Park offers services suc...,"Accessible Sites, Assistive Listening Systems,..."
26425,How much is a ticket for Great Egg Harbor River?,feespass,feespasses,greg,"Entrance to Great Egg Harbor River is free, as...",There is no entrance fee for Great Egg Harbor ...,Interagency Pass is not accepted.
28780,How much is a ticket for Great Egg Harbor River?,feespass,feespasses,greg,There is no entrance fee for Great Egg Harbor ...,There is no entrance fee for Great Egg Harbor ...,Interagency Pass is not accepted.
17754,What services are there at Washington Monument?,amenities,amenities,wamo,The Washington Monument is not a national park...,"I'm sorry, but I don't have information on the...","Bicycle - Rental, Gifts/Souvenirs/Books, Histo..."
16560,What services are there at Kenai Fjords Nation...,amenities,amenities,kefj,"At Kenai Fjords National Park, visitors can en...",Kenai Fjords National Park offers services suc...,"Accessible Sites, Assistive Listening Systems,..."
20585,What services are there at Washington Monument?,amenities,amenities,wamo,The Washington Monument is not a national park...,"I'm sorry, but I don't have information on the...","Bicycle - Rental, Gifts/Souvenirs/Books, Histo..."


In [61]:
def ragas_similarity(answer, context):
    """Converts answer and ground truth into a hugging face dataset so that answer similarity between the 2 can be calculated
    While RAGAS could be used to find the average answer similarities for the whole dataframe, this would not give case by case
        information on where the models are performing better or worse than one another etc.
    Input: anser = results returned from a model
        context = the context returned from API that should be compared against
    Output: score = the answer similarity score (cosine similarity) between answer and context as calculated by RAGAS """
    d1 = {
        "answer": [answer],
        "ground_truth": [context],
        }
    dataset1 = Dataset.from_dict(d1)
    scores = evaluate(dataset1, metrics=[answer_similarity])
    score = scores['answer_similarity']
    return score

In [30]:
def add_answer_similarity(query_df, chat_models, n=0):
    """ Adds answer similarity calculations to a dataframe of chat model responses and contexts (ground_truths)
    Input: query_df = dataframe 
        chat_models = list of model names (or dict with model names as keys) to be used for identifying columns to use as the model 'answers'
        n = counter for if this needs to be run in chunks or started part way through"""
    while n < len(query_df):
        segment_df = query_df[n:n+20]

        for model in chat_models:
            temp_df = segment_df.apply(lambda x: ragas_similarity(x[model],x['ground_truth']), axis=1)
            temp_df = pd.DataFrame(temp_df, columns = [f'{model}_sim'])
            segment_df = pd.concat([segment_df, temp_df], axis=1)
            segment_df = segment_df.fillna('none')
     
        if n ==0:
            test_df = segment_df
        else:
            test_df = pd.read_pickle('similarity.pkl')
            test_df = pd.concat([test_df, segment_df], axis=0)
            
        test_df.to_pickle('similarity.pkl')
        n=n+20
        print(n, 'completed')

In [77]:
##Uncomment to run if similarity.pkl does not exist. Otherwise takes >1 hour to run.
#add_answer_similarity(responses[:1100], ['lang_chat', 'gpt_chat'], 620)

In [ ]:
#Completed:
#0-100
#100-400 -- 8 min
#400-620
#620-1100

In [18]:
similarity = pd.read_pickle('similarity.pkl')
similarity

,query,intent,endpoint,parkcode,gpt_chat,lang_chat,ground_truth,lang_chat_sim,gpt_chat_sim
27641,What are the fees for visiting Virgin Islands ...,feespass,feespasses,vicr,There are no entrance fees for visiting Virgin...,Virgin Islands Coral Reef National Monument is...,Interagency Pass is not accepted.,0.735420,0.738262
15777,What is available at Denali National Park & Pr...,amenities,amenities,dena,"Denali National Park & Preserve, located in Al...",Denali National Park & Preserve offers a varie...,"Accessible Rooms, Assistive Listening Systems,...",0.903771,0.779647
24454,What events are scheduled at Lower Delaware?,events,events,lode,"I'm sorry, but there is no national park calle...",Lower Delaware National Wild and Scenic River ...,No events information found,0.748020,0.744912
6328,What is the address of Saint Paul's Church,address,parks,sapa,"I'm sorry, I can only provide information abou...",The address of Saint Paul's Church National Hi...,"897 South Columbus Avenue, Mount Vernon, NY, 1...",0.911092,0.709546
28771,What are the fees for visiting Grant-Kohrs Ranch,feespass,feespasses,grko,The entrance fee for Grant-Kohrs Ranch Nationa...,"Grant-Kohrs Ranch is a fee-free park, so there...",Interagency Pass is not accepted.,0.746890,0.745062
...,...,...,...,...,...,...,...,...,...
30025,How much is a ticket for Washington Monument?,feespass,feespasses,wamo,The Washington Monument is not a national park...,A ticket for the Washington Monument costs $1.00.,Interagency Pass is not accepted. There is no ...,0.882758,0.901474
20117,What services are there at Richmond Park?,amenities,amenities,rich,Richmond Park is a royal park located in Londo...,Richmond Park is a national park located in Lo...,No amenities information found,0.733472,0.736993
1660,Give me information on Theodore Roosevelt Inau...,description,parks,thri,Theodore Roosevelt Inaugural National Historic...,Theodore Roosevelt Inaugural National Historic...,"As president, Theodore Roosevelt created prote...",0.860418,0.864725
499,Give me information on Edgar Allan Poe Nationa...,description,parks,edal,"I'm sorry, but there is no national park calle...",The Edgar Allan Poe National Historic Site is ...,"Described as horrifying, mystifying, and brill...",0.890917,0.859263


In [19]:
len(similarity)

1100

In [20]:
similarity.duplicated(subset=['query']).value_counts()

False    1097
True        3
Name: count, dtype: int64

In [21]:
similarity.tail()

,query,intent,endpoint,parkcode,gpt_chat,lang_chat,ground_truth,lang_chat_sim,gpt_chat_sim
30025,How much is a ticket for Washington Monument?,feespass,feespasses,wamo,The Washington Monument is not a national park...,A ticket for the Washington Monument costs $1.00.,Interagency Pass is not accepted. There is no ...,0.882758,0.901474
20117,What services are there at Richmond Park?,amenities,amenities,rich,Richmond Park is a royal park located in Londo...,Richmond Park is a national park located in Lo...,No amenities information found,0.733472,0.736993
1660,Give me information on Theodore Roosevelt Inau...,description,parks,thri,Theodore Roosevelt Inaugural National Historic...,Theodore Roosevelt Inaugural National Historic...,"As president, Theodore Roosevelt created prote...",0.860418,0.864725
499,Give me information on Edgar Allan Poe Nationa...,description,parks,edal,"I'm sorry, but there is no national park calle...",The Edgar Allan Poe National Historic Site is ...,"Described as horrifying, mystifying, and brill...",0.890917,0.859263
14545,List any alerts for Puʻukoholā Heiau?,alerts,alerts,puhe,"As of now, there are no alerts or closures rep...",There are two alerts for Pu'ukoholā Heiau Nati...,Pu'ukoholā Heiau NHS Visitor Center and Museum...,0.961931,0.904686


Similarity Scores

In [75]:
lang_chat_avg = similarity['lang_chat_sim'].mean()
gpt_chat_avg = similarity['gpt_chat_sim'].mean()
print('Average Similarity score for LangChain Model with context: ', lang_chat_avg)
print('Average Similarity score for GPT Model with no context: ', gpt_chat_avg)



Average Similarity score for LangChain Model with context:  0.8639427709871653
Average Similarity score for GPT Model with no context:  0.8294172795530692


In [76]:
#Frequency Better
lang_better = len(similarity.query('lang_chat_sim>=gpt_chat_sim'))
gpt_beter = len(similarity) - lang_better
percent = (lang_better / len(similarity))*100
print(f'The Langchain Model performs better than the GPT model {lang_better} times')
print(f'The Langchain Model performs worse than the GPT model {gpt_beter} times')
print(f'The Langchain Model performs better than the GPT model in {percent}% of test cases')

The Langchain Model performs better than the GPT model 854 times
The Langchain Model performs worse than the GPT model 246 times
The Langchain Model performs better than the GPT model in 77.63636363636364% of test cases


In [25]:
intents = similarity.groupby('intent').agg({'lang_chat_sim': 'mean', 'gpt_chat_sim': 'mean'})
intents

,lang_chat_sim,gpt_chat_sim
intent,,
address,0.872028,0.851595
alerts,0.874505,0.807152
amenities,0.839540,0.795280
description,0.935322,0.899247
events,0.835112,0.804329
feespass,0.837072,0.818407
fullname,0.927400,0.924069
state,0.674771,0.672171


In [26]:
endpoints = similarity.groupby('endpoint').agg({'lang_chat_sim': 'mean', 'gpt_chat_sim': 'mean'})
endpoints

,lang_chat_sim,gpt_chat_sim
endpoint,,
alerts,0.874505,0.807152
amenities,0.839540,0.795280
events,0.835112,0.804329
feespasses,0.837072,0.818407
parks,0.896681,0.876552


In [27]:
parkcodes = similarity.groupby('parkcode').agg({'lang_chat_sim': 'mean', 'gpt_chat_sim': 'mean'})
parkcodes['diff'] = parkcodes['lang_chat_sim'] - parkcodes['gpt_chat_sim']
parkcodes.sort_values(by = 'diff', ascending=False).head()

,lang_chat_sim,gpt_chat_sim,diff
parkcode,,,
sapa,0.911092,0.709546,0.201546
kova,0.954950,0.758636,0.196314
tuin,0.969368,0.797505,0.171862
gree,0.934047,0.771159,0.162888
deto,0.946552,0.792275,0.154278


In [28]:
parkcodes.sort_values(by = 'diff', ascending=False).tail()

,lang_chat_sim,gpt_chat_sim,diff
parkcode,,,
nico,0.804178,0.838669,-0.034491
whho,0.734040,0.769413,-0.035373
bowa,0.786850,0.824431,-0.037581
mono,0.772792,0.811028,-0.038236
kowa,0.778689,0.828314,-0.049626


Time Comparison

In [28]:
timer = pd.read_pickle('chat_times.pkl')
mean_times = pd.DataFrame(timer.mean()).reset_index()
mean_times.columns = ['Model','Average Seconds per Call']
mean_times

,Model,Average Seconds per Call
0,gpt_chat,1.150051
1,lang_chat,1.670762
2,ground_truth,0.530556


#### Exploration
Examples of calls for each intent, comparing Context returned, LangChain Model with Context, and directly asking GPT model<br>
'lang_chat' is redefined to use GNB model underneath to simulate real world use<br>
'get_context' returns the direct API call, rather than the 'parsed_context' in evaluation above<br>

In [32]:
def lang_chat(query):
    """Takes Question as input and returns the result of running it through LangChain promp templat chat model (nps_chain.py) 
    Usesing get_context(query, model) for context as usual, the correct API landing page is handed through as context with 
    call_api() function which uses intent, endpoint and parkcode from test data to ensure that the correct context is referenced. 
    Returns model results"""
    return nps_chain.invoke({"context": get_context(query, model), "question": query})

##### Description

In [149]:
question = "Describe Yosemite"
get_context(question, model)

[{'id': '4324B2B4-D1A3-497F-8E6B-27171FAE4DB2',
  'url': 'https://www.nps.gov/yose/index.htm',
  'fullName': 'Yosemite National Park',
  'parkCode': 'yose',
  'description': 'Not just a great valley, but a shrine to human foresight, the strength of granite, the power of glaciers, the persistence of life, and the tranquility of the High Sierra. First protected in 1864, Yosemite National Park is best known for its waterfalls, but within its nearly 1,200 square miles, you can find deep valleys, grand meadows, ancient giant sequoias, a vast wilderness area, and much more.',
  'latitude': '37.84883288',
  'longitude': '-119.5571873',
  'latLong': 'lat:37.84883288, long:-119.5571873',
  'activities': [{'id': '09DF0950-D319-4557-A57E-04CD2F63FF42',
    'name': 'Arts and Culture'},
   {'id': '5F723BAD-7359-48FC-98FA-631592256E35', 'name': 'Auto and ATV'},
   {'id': '0B4A5320-216D-451A-9990-626E1D5ACE28', 'name': 'Scenic Driving'},
   {'id': '13A57703-BB1A-41A2-94B8-53B692EB7238', 'name': 'Astr

In [150]:
lang_chat(question)

'Yosemite National Park is not just a great valley, but a shrine to human foresight, the strength of granite, the power of glaciers, the persistence of life, and the tranquility of the High Sierra. First protected in 1864, Yosemite National Park is best known for its waterfalls, but within its nearly 1,200 square miles, you can find deep valleys, grand meadows, ancient giant sequoias, a vast wilderness area, and much more. It covers nearly 1,200 square miles in the Sierra Nevada, with elevations ranging from about 2,000 feet to 13,000 feet.'

In [151]:
gpt_chat(question)

"Yosemite National Park is located in California's Sierra Nevada mountains and is known for its stunning granite cliffs, waterfalls, giant sequoia trees, and diverse ecosystems. It is home to iconic landmarks such as El Capitan, Half Dome, and Yosemite Falls. The park offers a variety of outdoor activities including hiking, rock climbing, camping, and wildlife viewing. Yosemite is a popular destination for nature lovers and outdoor enthusiasts from around the world."

##### Location (address/state)

In [136]:
question = "Where is Yosemite"
get_context(question, model)

[{'id': '4324B2B4-D1A3-497F-8E6B-27171FAE4DB2',
  'url': 'https://www.nps.gov/yose/index.htm',
  'fullName': 'Yosemite National Park',
  'parkCode': 'yose',
  'description': 'Not just a great valley, but a shrine to human foresight, the strength of granite, the power of glaciers, the persistence of life, and the tranquility of the High Sierra. First protected in 1864, Yosemite National Park is best known for its waterfalls, but within its nearly 1,200 square miles, you can find deep valleys, grand meadows, ancient giant sequoias, a vast wilderness area, and much more.',
  'latitude': '37.84883288',
  'longitude': '-119.5571873',
  'latLong': 'lat:37.84883288, long:-119.5571873',
  'activities': [{'id': '09DF0950-D319-4557-A57E-04CD2F63FF42',
    'name': 'Arts and Culture'},
   {'id': '5F723BAD-7359-48FC-98FA-631592256E35', 'name': 'Auto and ATV'},
   {'id': '0B4A5320-216D-451A-9990-626E1D5ACE28', 'name': 'Scenic Driving'},
   {'id': '13A57703-BB1A-41A2-94B8-53B692EB7238', 'name': 'Astr

In [137]:
lang_chat(question)

'Yosemite National Park is located in California.'

In [138]:
gpt_chat(question)

'Yosemite National Park is located in California, USA. It covers an area of over 750,000 acres in the Sierra Nevada mountains.'

#### Events 

In [139]:
question = "What events are at Yosemite"
get_context(question, model)

[{'location': 'Glacier Point is located about 30 miles (48 km) or 1 hour from Yosemite Valley.',
  'updateuser': '',
  'contactname': '',
  'contacttelephonenumber': '',
  'recurrencedateend': '2024-08-20',
  'longitude': '',
  'datestart': '2024-08-07',
  'isrecurring': 'true',
  'datetimeupdated': '',
  'portalname': '',
  'types': ['Talk', 'Walk'],
  'createuser': '',
  'isfree': 'true',
  'contactemailaddress': '',
  'regresurl': '',
  'description': '<p>Acompaña a un guardaparques a aprender sobre Yosemite en Glacier Point. Domingo y miércoles, 2 pm a 3 pm.</p>',
  'images': [{'path': '/common/uploads/event_calendar/A89D7D8B-F4D7-CD65-04894ADDA6E8EFC5.jpg',
    'credit': 'NPS',
    'imageId': '43904',
    'altText': 'Arrowhead symbol for the National Park Service.',
    'title': '',
    'ordinal': '0',
    'caption': 'This program is hosted by the National Park Service',
    'url': '/common/uploads/event_calendar/A89D7D8B-F4D7-CD65-04894ADDA6E8EFC5.jpg'}],
  'category': 'Regular E

In [140]:
lang_chat(question)

'Here are some events happening at Yosemite National Park:\n\n1. ¡En español! Programa de Guardaparques (Glacier Point)\n2. Art Class (Yosemite Valley)\n3. Bird Walk (Tuolumne Meadows)\n4. Campfire Program (Wawona)\n5. Campfireside Chat with a Ranger (Tuolumne Meadows)\n6. Coffee With a Ranger (Wawona)\n7. Curry Village Historic Tour (Yosemite Valley)\n8. Discovery Hike: Vernal Fall Footbridge (Yosemite Valley)\n9. Dog Lake Hike with a Ranger (Tuolumne Meadows)'

In [142]:
gpt_chat(question)

'Yosemite National Park in California offers a variety of events throughout the year, including guided hikes, photography workshops, stargazing programs, art classes, and ranger-led talks. You can check the official Yosemite National Park website or contact the park directly for the most up-to-date information on events and activities.'

#### Address 

In [143]:
question = "What is address of everglades"
get_context(question, model)

[{'id': '5EA02193-276A-4037-B7DB-5765A56935FD',
  'url': 'https://www.nps.gov/ever/index.htm',
  'fullName': 'Everglades National Park',
  'parkCode': 'ever',
  'description': 'Everglades National Park protects an unparalleled landscape that provides important habitat for numerous rare and endangered species like the manatee, American crocodile, and the elusive Florida panther. An international treasure as well - a World Heritage Site, International Biosphere Reserve, a Wetland of International Importance, and a specially protected area under the Cartagena Treaty.',
  'latitude': '25.37294225',
  'longitude': '-80.88200301',
  'latLong': 'lat:25.37294225, long:-80.88200301',
  'activities': [{'id': '5F723BAD-7359-48FC-98FA-631592256E35',
    'name': 'Auto and ATV'},
   {'id': '0B4A5320-216D-451A-9990-626E1D5ACE28', 'name': 'Scenic Driving'},
   {'id': '13A57703-BB1A-41A2-94B8-53B692EB7238', 'name': 'Astronomy'},
   {'id': 'D37A0003-8317-4F04-8FB0-4CF0A272E195', 'name': 'Stargazing'},
 

In [144]:
lang_chat(question)

'The address of Everglades National Park is 40001 State Road 9336, Homestead, FL 33034, United States.'

In [145]:
gpt_chat(question)

'The Everglades National Park is located in Florida, and the main entrance address is 40001 State Road 9336, Homestead, FL 33034.'

#### Alerts 

In [146]:
question ='What alerts are at bryce?'
get_context(question, model)

[{'id': '9840308A-02FB-4A2F-90A3-E5597EB3881B',
  'url': 'https://www.nps.gov/brca/planyourvisit/conditions.htm',
  'title': 'Main Road Status',
  'parkCode': 'brca',
  'description': 'The main park road is fully open to Rainbow Point (Mile 18 of 18). All park roads are currently open for the season. During snowstorms the road may temporarily close at Mile 3 for snowplow operations.',
  'category': 'Information',
  'relatedRoadEvents': [],
  'lastIndexedDate': '2024-04-02 13:59:18.0'},
 {'id': '8833A24A-3F71-4E6E-9254-FF7743304115',
  'url': 'https://www.nps.gov/brca/planyourvisit/basicinfo.htm',
  'title': 'Bryce Canyon is Open - No Reservations Required to Enter',
  'parkCode': 'brca',
  'description': 'No reservations are required to enter Bryce Canyon National Park at any time of year. Simply pay your park entrance fee or present your America the Beautiful pass upon arrival.',
  'category': 'Information',
  'relatedRoadEvents': [],
  'lastIndexedDate': '2024-02-16 15:58:11.0'}]

In [147]:
lang_chat(question)

'There are currently no alerts at Bryce Canyon National Park. The main park road is fully open to Rainbow Point, and all park roads are currently open for the season.'

In [148]:
gpt_chat(question)

"At Bryce Canyon National Park, visitors should be aware of alerts related to weather conditions, trail closures, wildlife encounters, and safety reminders. It's always a good idea to check the park's official website or contact the visitor center for the most up-to-date information on any alerts or advisories in the park."

#### Fees

In [7]:
question ='What are the fees for grand canyon?'
get_context(question, model)

[{'parkCode': 'grca',
  'isFeeFreePark': False,
  'isInteragencyPassAccepted': True,
  'cashless': 'Yes',
  'feesAtWorkUrl': '',
  'entranceFeeDescription': "Admission to Grand Canyon National Park is for seven days and includes both the South Rim and during their season, the North Rim. No cash is accepted; credit/debit card only. No refunds are given due to inclement weather. Grand Canyon Annual Passes and America the Beautiful passes are available at all three of Grand Canyon National Park's entrance stations.",
  'entrancePassDescription': 'Grand Canyon National Park - Annual Pass Available to the general public for purchase for unlimited visits to Grand Canyon National Park only. This is an annual pass, valid one year from month of purchase; it is non-transferable. It admits the pass holder and any accompanying persons in a single, private, non-commercial vehicle, or the pass holder and accompanying immediate family (spouse, children, parents) when entry is by other means (train, s

In [12]:
lang_chat(question)

'The fees for Grand Canyon National Park are as follows:\n- Entrance fee for a private vehicle: $35.00\n- Entrance fee for a motorcycle: $30.00\n- Entrance fee per person (bicyclists, hikers, pedestrians): $20.00\n\nAdditionally, there is an option to purchase an Annual Pass for unlimited visits to Grand Canyon National Park for $70.00.'

In [13]:
gpt_chat(question)

'The entrance fee for Grand Canyon National Park is $35 per vehicle, which is valid for 7 days. There are also other pass options available, such as the annual pass for $70, which provides access to all national parks for one year. Additionally, there are discounts available for seniors, military members, and other groups.'

Full Name

In [29]:
question ='What is the full name of joshua park?'
get_context(question, model)

[{'id': 'F5CD58FB-05DC-4074-99DA-F327A537F1BC',
  'url': 'https://www.nps.gov/jotr/index.htm',
  'fullName': 'Joshua Tree National Park',
  'parkCode': 'jotr',
  'description': 'Two distinct desert ecosystems, the Mojave and the Colorado, come together in Joshua Tree National Park. A fascinating variety of plants and animals make their homes in a land sculpted by strong winds and occasional torrents of rain. Dark night skies, a rich cultural history, and surreal geologic features add to the wonder of this vast wilderness in southern California. Come explore for yourself!',
  'latitude': '33.91418525',
  'longitude': '-115.8398125',
  'latLong': 'lat:33.91418525, long:-115.8398125',
  'activities': [{'id': '5F723BAD-7359-48FC-98FA-631592256E35',
    'name': 'Auto and ATV'},
   {'id': '0B4A5320-216D-451A-9990-626E1D5ACE28', 'name': 'Scenic Driving'},
   {'id': '13A57703-BB1A-41A2-94B8-53B692EB7238', 'name': 'Astronomy'},
   {'id': 'D37A0003-8317-4F04-8FB0-4CF0A272E195', 'name': 'Stargazi

In [33]:
lang_chat(question)

'The full name of the park is Joshua Tree National Park.'

In [34]:
gpt_chat(question)

'The full name of Joshua Tree National Park is Joshua Tree National Park.'

Amenities

In [35]:
question ='What amenities are at bryce canyon?'
get_context(question, model)

[{'id': 'A1B0AD01-740C-41E7-8412-FBBEDD5F1443',
  'name': 'ATM/Cash Machine',
  'categories': ['Convenience', 'Souvenirs and Supplies']},
 {'id': '4E4D076A-6866-46C8-A28B-A129E2B8F3DB',
  'name': 'Accessible Rooms',
  'categories': ['Accessibility']},
 {'id': 'B509969B-F06A-4DCC-BBB3-134964D896E2',
  'name': 'Amphitheater',
  'categories': ['Interpretive', 'Recreation']},
 {'id': 'CBD31D2A-DD1A-4C5D-88C1-67B6B7147CE9',
  'name': 'Assistive Listening Systems',
  'categories': ['Accessibility']},
 {'id': '7DC6C690-2BC3-4344-9691-EF9FCFB6E506',
  'name': 'Audio Description',
  'categories': ['Accessibility']},
 {'id': 'F4A842CB-C966-402E-BBD3-AB86EDCF8403',
  'name': 'Automated Entrance',
  'categories': ['Accessibility']},
 {'id': 'B166EAD1-0D3E-4111-AE04-D5AB15DFDEDC',
  'name': 'Baby Changing Station',
  'categories': ['Convenience']},
 {'id': '20291812-E85C-4A84-B8A2-E31C8D149704',
  'name': 'Benches/Seating',
  'categories': ['Convenience']},
 {'id': '5BDFF349-2F74-490E-A33E-C3528615

In [36]:
lang_chat(question)

'Bryce Canyon National Park offers amenities such as benches/seating, information kiosk/bulletin board, parking - auto, picnic table, restroom, scenic view/photo spot, and trailhead.'

In [37]:
gpt_chat(question)

'Bryce Canyon National Park in Utah offers amenities such as campgrounds, lodging facilities, visitor centers, hiking trails, scenic overlooks, picnic areas, and ranger-led programs. There are also restaurants, gift shops, and restrooms available within the park.'

State

In [38]:
question ='What state is everglades in?'
get_context(question, model)

[{'id': '5EA02193-276A-4037-B7DB-5765A56935FD',
  'url': 'https://www.nps.gov/ever/index.htm',
  'fullName': 'Everglades National Park',
  'parkCode': 'ever',
  'description': 'Everglades National Park protects an unparalleled landscape that provides important habitat for numerous rare and endangered species like the manatee, American crocodile, and the elusive Florida panther. An international treasure as well - a World Heritage Site, International Biosphere Reserve, a Wetland of International Importance, and a specially protected area under the Cartagena Treaty.',
  'latitude': '25.37294225',
  'longitude': '-80.88200301',
  'latLong': 'lat:25.37294225, long:-80.88200301',
  'activities': [{'id': '5F723BAD-7359-48FC-98FA-631592256E35',
    'name': 'Auto and ATV'},
   {'id': '0B4A5320-216D-451A-9990-626E1D5ACE28', 'name': 'Scenic Driving'},
   {'id': '13A57703-BB1A-41A2-94B8-53B692EB7238', 'name': 'Astronomy'},
   {'id': 'D37A0003-8317-4F04-8FB0-4CF0A272E195', 'name': 'Stargazing'},
 

In [39]:
lang_chat(question)

'Everglades National Park is located in the state of Florida (FL).'

In [40]:
gpt_chat(question)

'Everglades National Park is located in the state of Florida.'